# StockBridge 🌉 - Voice-Powered Inventory Management

**Hackathon Submission Notebook**

StockBridge is a platform connecting shop owners with godown (warehouse) stock holders. Instead of typing stock updates or orders into forms, users **SPEAK** them in their own Indian language, and the app converts the speech into structured inventory actions.

## 1. Architecture Overview
- **Frontend:** React (Vite) + Tailwind CSS
- **Backend:** Node.js + Express
- **Database:** Prisma (MySQL / SQLite)
- **Real-time Sync:** Socket.io
- **AI Voice Parsing:** Google Gemini 2.5 Flash via `@google/genai`

Below is an interactive demonstration of our core AI logic: **Cross-lingual Intent Classification**. Judges can run this in SageMaker to test our voice-parsing logic live.

In [ ]:
!pip install google-genai pydantic

In [ ]:
import os
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import List, Optional

# Note to Judges: Insert your Gemini API Key here to test
os.environ['GEMINI_API_KEY'] = 'YOUR_API_KEY_HERE'

client = genai.Client()

In [ ]:
class Item(BaseModel):
    product_id: int
    qty: float
    unit: str
    confidence: float

class VoiceAction(BaseModel):
    intent: str = Field(description="STOCK_UPDATE, PLACE_ORDER, DISPATCH, RESTOCK_IN, or QUERY")
    confidence: float
    items: List[Item]
    target_shop_id: Optional[int]
    unmatched: List[str]
    clarification_needed: Optional[str]

PROMPT_TEMPLATE = """
You are processing a spoken inventory command for an Indian shop/godown system. 
The speaker may use Tamil, Hindi, Telugu, Kannada, Malayalam, Marathi, Bengali, English, or a natural mix.

Speaker role: {role}
Speaker said: "{transcript}"
Known products: {products}

Classify the intent and extract structured data.
"""

def parse_voice_command(transcript, role="shop_owner"):
    known_products = "[{'id': 1, 'name': 'Rice', 'unit': 'kg'}, {'id': 2, 'name': 'Oil', 'unit': 'litre'}]"
    prompt = PROMPT_TEMPLATE.format(
        role=role,
        transcript=transcript,
        products=known_products
    )
    
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=VoiceAction,
            ),
        )
        return response.text
    except Exception as e:
        return f"Error (Did you set the API key?): {e}"


In [ ]:
# Test Case 1: Tamil (Shop Owner ordering goods)
print(parse_voice_command("Enakku 20 kilo arisi, 10 litre oil venum"))

In [ ]:
# Test Case 2: Hindi (Godown Admin dispatching)
print(parse_voice_command("50 kilo chawal dispatch kar do shop 102 ko", role="godown_admin"))